# Walking UMA's manifold to a stabler green-hydrogen catalyst
Standalone Colab version. Use a GPU runtime (Runtime → Change runtime type → T4).

## 0. Setup
You need a free Hugging Face account with access to `facebook/UMA` (accept the licence on the model page, then create a read token).

In [ ]:
!pip install -q fairchem-core pymatgen scikit-learn

In [ ]:
from huggingface_hub import login; login()    # paste your Hugging Face token

In [ ]:
import urllib.request
for f in ['ruo2_110_slab.json', 'landscape_RuO2.json']:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/PranavViswanath/manifold-alchemy/main/data/' + f, f)

## 1. The method, in one cell
`VirtualUMA` lets any atom carry any element vector. `Chart` is the continuous periodic table. `walk` is gradient ascent on it.

In [ ]:
"""Minimal tools for walking UMA's element manifold.

UMA (fairchem, uma-s-1p2) turns each element into learned 128-d vectors in four lookup tables:
  sphere_embedding       initial node state           indexed by atom
  source_embedding       one end of every bond message indexed by edge source atom
  target_embedding       other end                     indexed by edge target atom
  composition_embedding  mean-pooled -> expert routing indexed by atom
`VirtualUMA` lets any atom carry ANY 128-d vector in all four tables, so element identity becomes continuous.
Energies are the model's raw per-atom-summed energy (no per-element reference offsets) so virtual atoms are well defined;
every quantity we use is an energy difference, where references cancel anyway.
"""
from __future__ import annotations
import numpy as np, torch, warnings
from scipy.interpolate import RBFInterpolator
from ase import Atoms
from ase.data import chemical_symbols
from pymatgen.core.periodic_table import Element
warnings.filterwarnings("ignore")

NORM = 1.342                      # UMA-S-1.2 energy normaliser: raw head output x NORM = eV
TABLES = ["sph", "src", "tgt", "comp"]


class VirtualUMA:
    """Load uma-s-1p2 and expose element identity as per-atom vectors that can be overridden."""

    def __init__(self, device="cuda"):
        from fairchem.core import pretrained_mlip, FAIRChemCalculator
        from fairchem.core.units.mlip_unit.api.inference import InferenceSettings
        self.Calc = FAIRChemCalculator
        settings = InferenceSettings(tf32=False, activation_checkpointing=False, merge_mole=False, compile=False, execution_mode="general")
        self.pred = pretrained_mlip.get_predict_unit("uma-s-1p2", device=device, inference_settings=settings)
        m = self.pred.model
        while hasattr(m, "module"):
            m = m.module
        self.bb = m.backbone
        self.head = m.output_heads["energyandforcehead"].head.energy_block
        self.mods = {"sph": self.bb.sphere_embedding, "src": self.bb.source_embedding, "tgt": self.bb.target_embedding, "comp": self.bb.composition_embedding}
        self.W = {k: self.mods[k].weight.detach().clone().to(device) for k in TABLES}   # the four element tables [100,128]
        self.dev = device; self.task = None; self.calc = None
        self.S = {"V": None, "ei": None, "E_raw": None, "net": None}
        self._ei_cache = {}
        self._install_hooks()

    # -- hooks: replace the four lookups by per-atom vectors when S["V"] is set -------------------------------------
    def _install_hooks(self):
        S = self.S
        def emb_hook(name):
            def h(mod, inp, out):
                V = S["V"]
                if V is None or V.get(name) is None:
                    return out
                v = V[name].to(out.device, out.dtype)
                if name in ("sph", "comp"):
                    return v                                        # [N,128]
                ei = S["ei"].to(out.device)
                return v[ei[0] if name == "src" else ei[1]]        # [E,128]
            return h
        for k in TABLES:
            self.mods[k].register_forward_hook(emb_hook(k))
        def ei_hook(mod, args, kw, out):                             # capture the graph on real-element passes
            if S["V"] is None:
                S["ei"] = (kw["edge_index"] if "edge_index" in kw else args[2]).detach()
        self.bb.blocks[0].edge_wise.register_forward_hook(ei_hook, with_kwargs=True)
        def energy_hook(mod, inp, out):                              # raw energy from the final scalar channels
            if S["net"] is not None:
                S["E_raw"] = float(S["net"](out["node_embedding"][:, 0, :]).sum())
        self.bb.register_forward_hook(energy_hook)

    def prime(self, task: str):
        """Select the level of theory (omat: bulk PBE; oc22: oxide surfaces; oc20: metal surfaces; omol: molecules)."""
        if self.task == task:
            return
        self.S["net"] = None
        a = Atoms("MgO", positions=[[0, 0, 0], [2.1, 0, 0]], cell=[8, 8, 8], pbc=True); a.info.update(charge=0, spin=1)
        a.calc = self.Calc(self.pred, task_name=task); a.get_potential_energy()
        eb = self.head
        self.S["net"] = torch.nn.Sequential(eb[0].merged_linear_layer(), torch.nn.SiLU(), eb[2].merged_linear_layer(), torch.nn.SiLU(), eb[4].merged_linear_layer())
        self.task = task; self.calc = self.Calc(self.pred, task_name=task)

    def _run(self, atoms):
        a = atoms.copy(); a.info.update(charge=0, spin=1)
        self.calc.reset()                                            # ASE caches identical atoms; embeddings changed
        a.calc = self.calc; a.get_potential_energy()
        return self.S["E_raw"] * NORM

    def default_V(self, atoms):
        """The real element vectors of each atom, as overridable per-atom tensors."""
        Z = torch.tensor(atoms.numbers, device=self.dev)
        return {k: self.W[k][Z].clone() for k in TABLES}

    def energy(self, atoms, V=None):
        """Energy in eV (reference-free). V = per-atom vectors from default_V, possibly edited with set_sites."""
        key = (atoms.numbers.tobytes(), np.round(atoms.positions, 5).tobytes(), np.round(atoms.cell.array, 5).tobytes())
        if V is None:
            self.S["V"] = None; E = self._run(atoms); self._ei_cache[key] = self.S["ei"]; return E
        if key not in self._ei_cache:
            while len(self._ei_cache) >= 64:
                self._ei_cache.pop(next(iter(self._ei_cache)))
            self.S["V"] = None; self._run(atoms); self._ei_cache[key] = self.S["ei"]
        self.S["ei"] = self._ei_cache[key]; self.S["V"] = V
        try:
            return self._run(atoms)
        finally:
            self.S["V"] = None


def set_sites(V, sites, vecs):
    """Give atoms `sites` the element vectors `vecs` (dict table -> 128-vector). Edits V in place."""
    for k in V:
        v = vecs[k]
        v = torch.as_tensor(np.asarray(v) if not torch.is_tensor(v) else v, dtype=V[k].dtype, device=V[k].device)
        V[k][list(sites)] = v
    return V


def delete_rows(V, idx):
    keep = [i for i in range(next(iter(V.values())).shape[0]) if i not in set(idx)]
    return {k: V[k][keep] for k in V}


def element_vectors(W, symbol):
    from ase.data import atomic_numbers
    return {k: W[k][atomic_numbers[symbol]] for k in TABLES}


# ---------------------------------------------------------------------------------------------------------------------
class Chart:
    """The continuous periodic table: a smooth map (group, period) -> element vectors, through the real elements.
    Thin-plate spline per table (Goodfire manifold-steering recipe: known concept coordinates, spline through centroids)."""

    def __init__(self, W, zmax=83):
        self.els = []
        for z in range(3, zmax + 1):
            e = Element(chemical_symbols[z])
            if 58 <= z <= 71 or e.group == 18:          # skip lanthanides and noble gases (little bonding data)
                continue
            self.els.append((z, float(e.group), float(e.row)))
        self.Z = np.array([z for z, _, _ in self.els]); self.gp = np.array([[g, p] for _, g, p in self.els])
        self.W = {k: (W[k].cpu().numpy() if hasattr(W[k], "cpu") else np.asarray(W[k])) for k in W}
        self.rbf = {k: RBFInterpolator(self.gp, self.W[k][self.Z], kernel="thin_plate_spline") for k in self.W}

    def vectors(self, g, p):
        return {k: self.rbf[k](np.array([[g, p]], float))[0] for k in self.W}

    def coords(self, symbol):
        e = Element(symbol); return float(e.group), float(e.row)

    def nearest(self, g, p):
        d = np.hypot(self.gp[:, 0] - g, self.gp[:, 1] - p); i = int(np.argmin(d))
        return chemical_symbols[self.Z[i]], float(d[i])


def walk(objective, start, chart, steps=8, h=0.25, step=0.5, decay=1.0, bounds=((2, 14), (4, 6)), log=print):
    """Gradient ascent of `objective(g, p)` on the chart by central finite differences. Returns the path."""
    g, p = chart.coords(start); path = [(g, p)]
    for it in range(steps):
        f0 = objective(g, p)
        dg = (objective(g + h, p) - objective(g - h, p)) / (2 * h)
        dp = (objective(g, p + h) - objective(g, p - h)) / (2 * h)
        n = np.hypot(dg, dp) + 1e-9; st = step * decay ** it
        g = float(np.clip(g + st * dg / n, *bounds[0])); p = float(np.clip(p + st * dp / n, *bounds[1]))
        path.append((g, p))
        log(f"step {it + 1}: f {f0:+.3f}  ->  (group {g:.2f}, period {p:.2f})  nearest {chart.nearest(g, p)[0]}")
    return path


In [ ]:
import numpy as np, json, matplotlib.pyplot as plt
from ase import Atoms
from ase.data import atomic_numbers, chemical_symbols

## 2. Load the model

In [ ]:
U = VirtualUMA('cuda')      # downloads uma-s-1p2 (~1 GB) on first use

## 3. Open it up: where do elements live?
Each element is a row of 128 learned numbers in four lookup tables.

In [ ]:
U.W.keys(), U.W['src'].shape

In [ ]:
W = {k: U.W[k].cpu().numpy() for k in TABLES}
W['src'][atomic_numbers['Fe']][:8]         # iron, first 8 of 128 numbers

Nearest neighbours in that space are chemical neighbours.

In [ ]:
def nearest(sym, table='src', n=4):
    X = W[table][1:84]; v = W[table][atomic_numbers[sym]]
    d = np.linalg.norm(X - v, axis=1); d[atomic_numbers[sym]-1] = np.inf
    return [chemical_symbols[i+1] for i in np.argsort(d)[:n]]
for s in ['Fe', 'Au', 'Cl', 'Na', 'Ti']: print(s, '->', nearest(s))

## 4. The static periodic table, as UMA drew it

In [ ]:
from sklearn.decomposition import PCA
Zs = [z for z in range(3, 84) if not 58 <= z <= 71]
X = np.concatenate([W['src'][Zs], W['tgt'][Zs]], 1)
P = PCA(3).fit(X); S = P.transform(X); P.explained_variance_ratio_.round(2)

In [ ]:
from pymatgen.core.periodic_table import Element
block = [Element(chemical_symbols[z]).block for z in Zs]
col = {'s': '#E07B39', 'p': '#3B9AB2', 'd': '#6C4AB6', 'f': 'grey'}
plt.figure(figsize=(9, 6), facecolor='#FAF6EE')
plt.scatter(S[:, 0], S[:, 1], c=[col[b] for b in block], s=60)
for (x, y), z in zip(S[:, :2], Zs): plt.annotate(chemical_symbols[z], (x, y), fontsize=8)
plt.xlabel('learned axis 1 (~row / size)'); plt.ylabel('learned axis 2 (~electronegativity / group)'); plt.title("UMA's periodic table");

Three learned axes carry most of the table, and they name to row, size and electronegativity. The model rediscovered Mendeleev from energies alone.

## 5. Making element identity continuous
First check: a real vector reproduces the real energy.

In [ ]:
U.prime('omat')
mgo = Atoms('MgO', positions=[[0, 0, 0], [2.1, 0, 0]], cell=[8, 8, 8], pbc=True)
E_mg = U.energy(mgo)
cao = mgo.copy(); cao.numbers[0] = atomic_numbers['Ca']; E_ca = U.energy(cao)

In [ ]:
V = U.default_V(mgo); set_sites(V, [0], element_vectors(U.W, 'Ca'))
E_virtual = U.energy(mgo, V)
print(f'real Ca {E_ca:.4f}   virtual Ca {E_virtual:.4f}')

Now a point *between* Mg and Ca. It has an energy. It is not a real element, and not a physical mixture either. What matters is the slope.

In [ ]:
half = {k: 0.5 * (U.W[k][12] + U.W[k][20]) for k in TABLES}
V = U.default_V(mgo); set_sites(V, [0], half); print('halfway Mg->Ca:', round(U.energy(mgo, V), 4), ' linear average:', round(0.5 * (E_mg + E_ca), 4))

## 6. The continuous periodic table (the chart)
A smooth surface through the real elements in (group, period) coordinates.

In [ ]:
CH = Chart(U.W)
CH.coords('Ru'), CH.nearest(5.5, 6.0)

In [ ]:
v = CH.vectors(5.5, 6.0)      # a point between Ta and W
{k: v[k][:3].round(3) for k in v}

## 7. The problem: green hydrogen needs a cheaper anode
PEM electrolysers use IrO₂ because RuO₂, cheaper and more active, dissolves in acid. It dissolves when its lattice oxygen next to a Ru site gets pulled into the reaction. Dopants that make that oxygen harder to remove are the field's fix (Ta, W, Nb, Ti... found one at a time over a decade).

Load a RuO₂(110) surface. `i` is the most exposed Ru; `nbrs` are the Ru positions a dopant could occupy; `O3` are the three oxygens bonded to the site.

In [ ]:
d = json.load(open('ruo2_110_slab.json'))
slab = Atoms(numbers=d['numbers'], positions=d['positions'], cell=d['cell'], pbc=True)
i, nbrs = d['i'], d['nbrs']
zz = slab.positions[:, 2]; dist = slab.get_distances(i, range(len(slab)), mic=True)
O3 = [int(k) for k in np.argsort(dist) if slab.numbers[k] == 8 and zz[k] > zz.mean() - 1 and dist[k] < 2.6][:3]
slab.get_chemical_formula(), i, nbrs, O3

## 8. The objective: how tightly is the lattice oxygen held?
Mean energy to remove each of the three oxygens bonded to the site. Higher = more protected.

In [ ]:
U.prime('oc22')
def protection(V=None):
    E = U.energy(slab, V); vals = []
    for k in O3:
        s = slab.copy(); del s[k]
        vals.append(U.energy(s, None if V is None else delete_rows(V, [k])) - E)
    return float(np.mean(vals))
base = protection(); base

## 9. One dopant, the old way: substitute and recompute

In [ ]:
j = 22
def exact(sym):
    s = slab.copy(); s.numbers[j] = atomic_numbers[sym]
    E = U.energy(s); vals = []
    for k in O3:
        t = s.copy(); del t[k]; vals.append(U.energy(t) - E)
    return float(np.mean(vals)) - base
print('Ta', round(exact('Ta'), 3), '  Cu', round(exact('Cu'), 3))

## 10. The alchemical gradient: which way is uphill?

In [ ]:
def f(g, p):
    V = U.default_V(slab); set_sites(V, [j], CH.vectors(g, p)); return protection(V) - base
g0, p0 = CH.coords('Ru'); h = 0.25
slope = ((f(g0 + h, p0) - f(g0 - h, p0)) / (2 * h), (f(g0, p0 + h) - f(g0, p0 - h)) / (2 * h))
slope

Negative in group, positive in period: fewer d-electrons, heavier row. That is the field's 'high-valence dopant' rule, as a derivative.

## 11. The landscape
The objective over the whole continuous table (~20 min on a GPU; a cached copy is loaded by default).

In [ ]:
RECOMPUTE = False
if RECOMPUTE:
    grid = {(g, p): f(g, p) for g in np.arange(2, 14.01, 0.5) for p in np.arange(4, 6.01, 0.5)}
else:
    L = json.load(open('landscape_RuO2.json')); grid = {tuple(float(x) for x in k.split(',')): v for k, v in L['grid'].items()}
len(grid)

In [ ]:
gs = sorted({g for g, _ in grid}); ps = sorted({p for _, p in grid}); Z = np.array([[grid[(g, p)] for g in gs] for p in ps])
plt.figure(figsize=(11, 3.5), facecolor='#FAF6EE'); plt.imshow(Z, origin='lower', extent=[2, 14, 4, 6], aspect='auto', cmap='turbo')
plt.colorbar(label='lattice-O protection (eV)'); plt.xlabel('group'); plt.ylabel('period')
for s in ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Nb', 'Mo', 'Ru', 'Rh', 'Pd', 'Ag', 'Ta', 'W', 'Re', 'Ir', 'Pt', 'Au']:
    g, p = CH.coords(s); plt.plot(g, p, 'k.'); plt.annotate(s, (g, p), color='w', fontsize=8)
plt.title('one ridge along groups 5-6, rising toward period 6');

## 12. The walk
Gradient ascent on the chart, starting from Ru.

In [ ]:
path = walk(f, 'Ru', CH, steps=8)

In [ ]:
g_end, p_end = path[-1]; CH.nearest(g_end, p_end)

## 13. Snap to a real element and confirm

In [ ]:
print('Ta', round(exact('Ta'), 3), ' W', round(exact('W'), 3))

In [ ]:
CANDS = 'Mg Ca Sr Ba Al Ga In Sn Sb Bi Sc Y La Ti Zr Hf V Nb Ta Cr Mo W Mn Fe Co Ni Cu Zn Rh Pd Ir Pt Ag Au Ge Si'.split()
ranking = sorted(((exact(s), s) for s in CANDS), reverse=True)
ranking[:6], ranking[-4:]

## 14. What the literature says
Ta- and W-doped RuO₂ are established acid-OER stabilisers: Sr/Ta co-doped RuO₂ at 166 mV (Nat. Commun. 2025), Ta/B–RuO₂ (Nat. Commun. 2025), W-doped RuO₂ (multiple reports, 2024–25). The walk recovered them from the model's own geometry, with no chemistry supplied.

## 15. Going further
* Second derivatives along the manifold predict *all* substitutions from one structure (445 sites: ionic 0.16→0.63, O 0.72→0.90).
* Cross-site curvature predicts co-doping pairs (0.97 on 45 exact checks; Ta/W doubles the best single dopant).
* The same derivative at the oxygen site points to fluorine, the experimentally proven anion dopant.

![walk](https://raw.githubusercontent.com/PranavViswanath/manifold-alchemy/main/media/manifold_walk.gif)